In [1]:
import bs4
import requests
import pandas as pd
from tqdm.notebook import tqdm
tqdm.pandas()
import warnings
warnings.filterwarnings('ignore')
import urllib
import io
import re
import os

In [14]:
def search_book(query: str) -> str:
    return requests.get('https://libgen.li/index.php',
         params = {
            'lg_topic': 'libgen',
            'open': 0,
            'view': 'simple',
            'res': 25,
            'phrase': 1,
            'column': 'def',
            'req': query
        }, headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:101.0) Gecko/20100101 Firefox/101.0",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.5",
        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "same-origin",
        "Sec-Fetch-User": "?1"
    }, verify=False)

def get_dl_link(mirror: str) -> str: 
    # print(mirror)
    res = requests.get(mirror)
    dl_page = bs4.BeautifulSoup(res.text, 'lxml')
    return dl_page.find('div', {'id': 'download'}).find('a')['href']

def download_file(link: str, name: str):
    print(f'downloading to {name}')
    filename, _ = urllib.request.urlretrieve(link, name)
    return filename

def get_download_options(book: str):
    cols = ['ID', 'Author(s)', 'Title', 'Publisher', 'Year', 'Pages', 'Language', 'Size', 'Extension', 'Mirrors', 'dl_link', 'book']
    search_result = search_book(book)
    print(search_result.text)
    soup = bs4.BeautifulSoup(search_result.text, "lxml")
    raw_table = soup.findAll('table', {'class': 'c'})
    if len(raw_table) == 0:
        return pd.read_csv(io.StringIO(','.join(cols)))
    for a in raw_table[0].findAll('a', {'title': 'Libgen & IPFS & Tor'}):
        # print('adding', a['href'])
        a.string = a['href']
    # print(raw_table)
    df = pd.read_html('<html><body>' + str(raw_table) + '</body></html>', header=0)[0]
    # print(df.head())
    df = df.assign(dl_link=df['Mirrors'].apply(get_dl_link), book=book)
    return df[df.Extension.isin(['epub', 'mobi'])][cols]


In [11]:
books = '''a thousand splendid suns'''.split('\n')

In [15]:
book_options = list(map(get_download_options, tqdm(books)))

  0%|          | 0/1 [00:00<?, ?it/s]

<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Transitional//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-transitional.dtd">
<html xmlns="http://www.w3.org/1999/xhtml">
<head>
	<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />
	<META HTTP-EQUIV="CACHE-CONTROL" CONTENT="max-age=604800, must-revalidate">
	<meta name="rating" content="general">
	<!--<link href="/rss/index.php" rel="alternate" type="application/rss+xml" title="News" />-->
	<link rel="shortcut icon" href="/img/favicon.ico" type="image/x-icon">
	<title>Library Genesis</title>
		
	<!--[if IE 6]>
	<style>
		body {behavior: url("/csshover3.htc");}
		#menu li .drop {background:url("img/drop.gif") no-repeat right 8px; 
	</style>
	<![endif]-->
<link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap@4.5.3/dist/css/bootstrap.min.css">	
	
<link href="/css/font.min.css" rel="stylesheet">	
<style>
nav.navbar .dropdown:hover > .dropdown-menu {
 display: block; 
}
.bd-placeholder-img {
	font-size: 1.125rem;
	

In [17]:
book_options_df = pd.concat(book_options).reset_index(drop=True)
book_options_df = book_options_df[book_options_df.Language.isin(["English"])]

In [18]:
book_options_df

,ID,Author(s),Title,Publisher,Year,Pages,Language,Size,Extension,Mirrors,dl_link,book


In [8]:
book_options_df.to_csv('book_options_20240310.csv', header=True, index=False)

In [12]:
# And now we wait

In [5]:
books_dl = pd.read_csv('book_options_20240310.csv', encoding = "ISO-8859-1")

In [6]:
DL_ALL = 'Y'
books_to_dl = books_dl.assign(
    filename=books_dl.Title.apply(
        lambda x: ''.join([i for i in re.sub('(\d{2,}(,)?)?[-,:\?]?', '', x)\
                           .replace('-', '').replace(',', '').replace(':', '').lstrip().rstrip() if ord(i) < 128])+'.epub'),
    Wanted = lambda x: books_dl.Wanted if 'Wanted' in books_dl.columns else DL_ALL
)[lambda _df: _df.Wanted.str.lower() == 'y'][['filename', 'dl_link']].reset_index(drop=True)

In [7]:
books_to_dl

,filename,dl_link
0,Remember Who You Are What Pedro Gomez Showed U...,https://download.library.lol/main/3580000/8def...
1,The Murder of King Tut The Plot to Kill the Ch...,https://download.library.lol/main/491000/c088f...
2,Anatomy of Injustice A Murder Case Gone Wrong ...,https://download.library.lol/main/2320000/8dc9...
3,Iphigenia in Forest Hills Anatomy of a Murder ...,https://download.library.lol/main/2662000/879b...
4,Where the Gods Dwell Thirteen Temples and thei...,https://download.library.lol/main/3164000/1aa9...
...,...,...
64,In The Woods.epub,https://download.library.lol/main/807000/295c7...
65,Lady Julia Grey Series Book 1 Lady Julia Grey ...,https://download.library.lol/main/589000/4acc3...
66,Silent On The Moor (A Lady Julia Grey Novel) [...,https://download.library.lol/main/732000/27dec...
67,The Dark Enquiry [Originaled.].epub,https://download.library.lol/main/758000/92967...


In [10]:
print(*sorted(books_to_dl.filename), sep='\n')

A Spark of Light [Hardcovered.].epub
Abraham Verghese. A Biography.epub
All the Ever Afters The Untold Story of Cinderellas Stepmother.epub
Anatomy of Injustice A Murder Case Gone Wrong [1ed.].epub
Breaking the Silence [Originaled.].epub
Crossing the Lines.epub
Darcy Lott 3 Civil Twilight A Darcy Lott Mystery.epub
Faithful Place.epub
Faithful Place.epub
Finding Chika A Little Girl An Earthquake And The Making Of A Family.epub
Halfway to the Grave (Night Huntress Series 1).epub
Handle with Care A Novel.epub
Have a Little Faith A True Story [Reprinted.]  X.epub
Homemade Body Butters Simple Recipes for Soft Healthy and Beautiful Looking Skin.epub
In The Woods.epub
Iphigenia in Forest Hills Anatomy of a Murder Trial.epub
Katherine Mansfield The StoryTeller.epub
Keeping Faith A Novel [Reprinted.].epub
Lady Julia Grey Series Book 1 Lady Julia Grey Series   Silent on the Moor.epub
Leaving Time A Novel by Jodi Picoult | Digest & Review.epub
Life is a gift  the zen of Bennett [1st ed].epub
Merc

In [ ]:
books_to_dl[14:].drop_duplicates('filename').progress_apply(lambda x: download_file(x.dl_link, 
                                          os.path.join('/home/cptfoobar/Documents',
                                                      x.filename)), axis=1)

  0%|          | 0/51 [00:00<?, ?it/s]

downloading to /home/cptfoobar/Documents/The Art of the StoryTeller.epub
downloading to /home/cptfoobar/Documents/Rise of the Truth Teller Own Your Story Tell It Like It Is and Live with Holy Gumption.epub


# 